# Setup

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
%cd Hierarchical-Localization
!pip install --progress-bar off --quiet -e .
!pip install --progress-bar off --quiet --upgrade plotly
# only build with ONNX gpu + caspar + ceres cuda/cudss; downloads disabled, so model paths must be set
!pip uninstall -y --quiet pycolmap
!pip install --progress-bar off --quiet \
   "https://github.com/lyehe/build_gpu_colmap/releases/download/v4.1.0/pycolmap-4.1.0+cu128.bundled.cudss-cp312-cp312-manylinux_2_35_x86_64.whl"
!pip install --progress-bar off --quiet mcap av


In [ ]:
from tqdm import tqdm
from pathlib import Path
import json
import numpy as np
import os
import shutil
import urllib.request


from hloc import reconstruction, visualization
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d
import pycolmap

In [4]:
DATA_PATH = Path('/kaggle/input/datasets/yu5uf5/buggy-hloc')
outputs = Path('/kaggle/working/multi')
outputs.mkdir(exist_ok=True)
IMAGES_PATH = Path('/kaggle/working/images')

Two data layouts are supported side by side:

- **old**: `<run>.mp4` + `vid_imu/<run>.json` (gopro video + racebox/fit export)
- **robo**: `<folder>/vid.svo2` + `<folder>/poses.json` (ZED 2i recording + GQ7 EKF poses). Generate `poses.json` locally with `export_robo_poses.py`, which also recovers the video→bag clock offset from gyro cross-correlation and converts ellipsoid heights to the course DEM datum shared with the old data. Runs are named `r<folder>`; both eyes are used as a COLMAP sensor rig with extrinsics from the factory calibration.

In [5]:
# old format
videos = {v.stem: v for v in DATA_PATH.glob('*.mp4')}
data = {}
for p in DATA_PATH.glob('vid_imu/*.json'):
    with open(p) as f:
        data[p.stem] = json.load(f)

# robo format
robo = {}
for p in sorted(DATA_PATH.glob('*/vid.svo2')):
    pose_file = p.parent / 'poses.json'
    if pose_file.exists():
        with open(pose_file) as f:
            robo[f'r{p.parent.name}'] = {'svo': p, **json.load(f)}

print(f'old runs: {sorted(data)}  robo runs: {sorted(robo)}')


# Create Images

In [ ]:
import cv2

STRIDE = 1  # keep every Nth frame
sharpness = {}  # run -> {ts_ns: variance of laplacian}, scored on save

def sharpness_score(bgr):
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (426, 240))
    return float(cv2.Laplacian(g, cv2.CV_32F).var())

In [12]:
for stem, video in tqdm(videos.items(), desc='videos'):
    start_ns = data[stem]['camera_start']  # video start, nanoseconds
    out_dir = IMAGES_PATH / stem
    out_dir.mkdir(parents=True, exist_ok=True)
    sharpness[stem] = {}
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved = 0
    i = 0
    try:
        with tqdm(total=frame_count, desc=stem, leave=False) as progress:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                progress.update(1)
                if i % STRIDE != 0:
                    continue
                timestamp_ns = start_ns + int(round(cap.get(cv2.CAP_PROP_POS_MSEC) * 1_000_000))
                cv2.imwrite(str(out_dir / f"{timestamp_ns}.jpg"), frame)
                sharpness[stem][timestamp_ns] = sharpness_score(frame)
                saved += 1
    finally:
        cap.release()
    print(f"{stem}: saved {saved} frames to {out_dir}")

In [ ]:
import av
from mcap.reader import make_reader

# svo2 is an mcap container of side-by-side H.264; split into both eyes, name frames by
# bag-clock epoch ns (sync offset applied), drop frames outside pose coverage
for run, r in robo.items():
    dirs = {side: IMAGES_PATH / run / side for side in ('left', 'right')}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    sharpness[run] = {}
    off_ns = int(round(r['sync']['offset_s'] * 1e9))
    t0_ns, t1_ns = int(r['ekf']['t'][0] * 1e9), int(r['ekf']['t'][-1] * 1e9)
    codec = av.CodecContext.create('h264', 'r')
    saved = skipped = errs = i = 0
    with open(r['svo'], 'rb') as f:
        reader = make_reader(f)
        with tqdm(total=r['camera']['num_frames'], desc=run, leave=False) as progress:
            for _, ch, msg in reader.iter_messages():
                if not ch.topic.endswith('/side_by_side'):
                    continue
                progress.update(1)
                try:
                    frames = [fr for pkt in codec.parse(msg.data[8:]) for fr in codec.decode(pkt)]
                except av.error.InvalidDataError:  # corrupt message; decode recovers at next keyframe
                    errs += 1
                    continue
                for fr in frames:
                    i += 1
                    if i % STRIDE != 0:
                        continue
                    ts_ns = msg.log_time + off_ns
                    if not (t0_ns <= ts_ns <= t1_ns):
                        skipped += 1
                        continue
                    img = fr.to_ndarray(format='bgr24')
                    w = img.shape[1] // 2
                    cv2.imwrite(str(dirs['left'] / f'{ts_ns}.jpg'), img[:, :w])
                    cv2.imwrite(str(dirs['right'] / f'{ts_ns}.jpg'), img[:, w:])
                    sharpness[run][ts_ns] = sharpness_score(img[:, :w])
                    saved += 1
    print(f'{run}: saved {saved} stereo pairs, {skipped} outside pose coverage, {errs} corrupt messages')


# Config

In [ ]:
sfm_pairs = outputs / 'pairs-sfm.txt'
loc_pairs = outputs / 'pairs-loc.txt'
sfm_dir = outputs / 'sfm'
sfm_prior_dir = outputs / 'sfm_prior'
database = outputs / 'database.db'

In [ ]:
# racebox: 25Hz + scalar doppler speed; fit: 10Hz + velocity vectors
GPS_KIND = 'racebox'
LAT0, LON0, ALT0 = 40.44163016, -79.94165829, 288.42151354  # shared ENU reference

DELTA_S = 2.0         # m between selected frames
SEQ_K = 6             # forward sequential pairs per frame
CROSS_K = 3           # nearest cross-run candidates per frame
CROSS_R = 6.0         # m, max cross-run pair distance
HEADING_MAX_DEG = 40  # max cross-run heading difference

# robo runs: camera sits a few cm in front of the gnss_2 antenna (+x body)
CAM_FROM_GNSS2 = np.array([0.05, 0.0, -0.01])  # m, tune once measured
ROBO_PRIOR_VAR_FLOOR = 0.1**2  # m^2, don't trust EKF covariance below this
OLD_CAM_HEIGHT = 0.5  # m, old runs' alt is DEM road level; lift priors to the camera


In [ ]:
import torch

GPU_INDEX = ','.join(str(i) for i in range(torch.cuda.device_count()))

# caspar BA only supports SIMPLE_RADIAL [f, cx, cy, k1]/PINHOLE
# initial estimates refined per image
CAMERA_PARAMS = [653.4, 631.72, 338.74, -0.0526]  # base parameters for virb
ZED_FALLBACK_PARAMS = [952.5, 661.5, 374.6, -0.075]

def zed_camera_params(conf, section='LEFT_CAM_HD'):
    # SIMPLE_RADIAL seed from the factory calibration (unrectified left @ HD720)
    if not conf:
        return ZED_FALLBACK_PARAMS
    cur, vals = None, {}
    for line in conf.splitlines():
        line = line.strip()
        if line.startswith('['):
            cur = line.strip('[]')
        elif '=' in line and cur == section:
            k, v = line.split('=')
            vals[k] = float(v)
    return [(vals['fx'] + vals['fy']) / 2, vals['cx'], vals['cy'], vals['k1']]

ZED_CAMERA_PARAMS = {
    run: {side: zed_camera_params(r['camera']['factory_calibration_conf'], f'{side.upper()}_CAM_HD')
          for side in ('left', 'right')}
    for run, r in robo.items()}


# Mapping

## Select Frames

In [ ]:
runs = sorted(set(data) | set(robo), key=lambda r: int(r.lstrip('r')))
gps_field = 'racebox_gps' if GPS_KIND == 'racebox' else 'gps_data'
gt = pycolmap.GPSTransform(pycolmap.GPSTransfromEllipsoid.WGS84)

def gps_arrays(run):
    gps = data[run][gps_field]
    return tuple(np.array([s[k] for s in gps], float) for k in ('timestamp', 'lat', 'long', 'alt'))

def speed_arrays(run):
    if GPS_KIND == 'racebox':
        spd = data[run]['racebox_speed']
        return (np.array([s['timestamp'] for s in spd], float),
                np.array([s['speed'] for s in spd], float))
    vel = data[run]['velocity']
    return (np.array([s['timestamp'] for s in vel], float),
            np.hypot([s['vx'] for s in vel], [s['vy'] for s in vel]))

def quat_rotate(q, v):
    # q (n,4) xyzw, v (3,) -> (n,3)
    qv, w = q[:, :3], q[:, 3:]
    t = 2 * np.cross(qv, v)
    return v + w * t + np.cross(qv, t)

def ecef_to_enu_rot(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    sla, cla, slo, clo = np.sin(la), np.cos(la), np.sin(lo), np.cos(lo)
    return np.array([[-slo, clo, 0.0],
                     [-sla * clo, -sla * slo, cla],
                     [cla * clo, cla * slo, sla]])

def run_pos_speed(run):
    """(pos_ts_ns, camera enu, speed_ts_ns, speed m/s)"""
    if run in robo:
        r = robo[run]
        e = r['ekf']
        ts = np.array(e['t']) * 1e9
        lla = np.stack([e['lat'], e['lon'], e['alt']], 1)
        enu = np.array(gt.ellipsoid_to_enu(list(lla), LAT0, LON0, ALT0))
        # ekf pose is of imu_link; move to the camera with the body lever arm
        lever = np.array(r['tf']['imu_link->gnss_2_antenna_link']) + CAM_FROM_GNSS2
        enu += quat_rotate(np.array(e['quat']), lever) @ ecef_to_enu_rot(LAT0, LON0).T
        return ts, enu, ts, np.array(e['speed'])
    gps_ts, lat, lon, alt = gps_arrays(run)
    enu = np.array(gt.ellipsoid_to_enu(list(np.stack([lat, lon, alt], 1)), LAT0, LON0, ALT0))
    enu[:, 2] += OLD_CAM_HEIGHT
    spd_ts, spd = speed_arrays(run)
    return gps_ts, enu, spd_ts, spd

sel = {}
run_enu = {}
for run in runs:
    img_dir = (IMAGES_PATH / run / 'left') if run in robo else (IMAGES_PATH / run)
    image_paths = sorted(img_dir.glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    pos_ts, enu_src, spd_ts, spd = run_pos_speed(run)
    run_enu[run] = (pos_ts, enu_src)

    # distance-uniform selection over the video ∩ pose window
    m = (spd_ts >= max(pos_ts[0], image_ts[0])) & (spd_ts <= min(pos_ts[-1], image_ts[-1]))
    ts_w, v_w = spd_ts[m], spd[m]
    dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
    sel_ts = np.interp(np.arange(0.0, dist[-1], DELTA_S), dist, ts_w)

    # sharpest frame within each target's window (blur is vibration-driven, varies frame to frame)
    scores = sharpness.get(run)
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2],
                             (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
        elif scores:
            idx.append(i0 + int(np.argmax([scores.get(int(u), 0.0) for u in image_ts[i0:i1]])))
        else:
            idx.append(i0 + np.abs(image_ts[i0:i1] - t).argmin())
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, pos_ts, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    sel[run] = {
        'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
        'enu': enu,
        'heading': np.arctan2(grad[:, 1], grad[:, 0]),
    }
    print(f'{run}: {len(idx)} frames over {dist[-1]:.0f}m')

references = [n for run in runs for n in sel[run]['names']]
len(references)


In [ ]:
import matplotlib.pyplot as plt

for run in runs:
    plt.plot(*sel[run]['enu'][:, :2].T, '.', ms=2, label=run)
plt.axis('equal')
plt.legend(markerscale=5)
plt.show()

In [ ]:
sl = slice(200, 210)
plot_images([read_image(IMAGES_PATH / ref) for ref in references[sl]],
            titles=references[sl], dpi=25)

## Features

In [ ]:
# this pycolmap build can't auto-download onnx models; stage them locally
MODELS_PATH = Path('/kaggle/working/models')
MODELS_PATH.mkdir(exist_ok=True)
for name in ('aliked-n16rot.onnx', 'aliked-lightglue.onnx'):
    f = MODELS_PATH / name
    if not f.exists():
        urllib.request.urlretrieve(
            f'https://github.com/colmap/colmap/releases/download/3.13.0/{name}', f)


# colmap masks: <mask_path>/<image name>.png, black = exclude
# per-run mask DATA_PATH/mask_<run>.png; old runs fall back to the shared mask0.png
MASKS_PATH = Path('/kaggle/working/masks')
MASKS_PATH.mkdir(exist_ok=True)
for run in runs:
    src = DATA_PATH / f'mask_{run}.png'
    if not src.exists():
        src = DATA_PATH / 'mask0.png' if run in data else None
    if src is None:
        continue
    staged = MASKS_PATH / src.name
    if not staged.exists():
        shutil.copy(src, staged)
    refs = sel[run]['names']
    if run in robo:
        refs = refs + [n.replace('/left/', '/right/') for n in refs]
    for ref in refs:
        dst = MASKS_PATH / f'{ref}.png'
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.link(staged, dst)


In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, outputs / 'colmap.LOG.')
database.unlink(missing_ok=True)

extraction_options = pycolmap.FeatureExtractionOptions()
extraction_options.type = pycolmap.FeatureExtractorType.ALIKED_N16ROT
extraction_options.aliked.n16rot_model_path = str(MODELS_PATH / 'aliked-n16rot.onnx')
extraction_options.aliked.max_num_features = 4096
extraction_options.use_gpu = True
extraction_options.gpu_index = GPU_INDEX

# one extraction per camera-param group: all old runs share a seed, robo runs seed
# from their own factory calibration
groups = []
old_names = [n for run in runs if run in data for n in sel[run]['names']]
if old_names:
    groups.append((CAMERA_PARAMS, old_names))
for run in runs:
    if run in robo:
        names_l = sel[run]['names']
        groups.append((ZED_CAMERA_PARAMS[run]['left'], names_l))
        groups.append((ZED_CAMERA_PARAMS[run]['right'], [n.replace('/left/', '/right/') for n in names_l]))

for params, names in groups:
    pycolmap.extract_features(
        database, IMAGES_PATH, image_names=sorted(names),
        camera_mode=pycolmap.CameraMode.PER_FOLDER,  # consider PER_IMAGE because of stabilization
        reader_options={'camera_model': 'SIMPLE_RADIAL',
                        'camera_params': ','.join(str(v) for v in params),
                        'mask_path': str(MASKS_PATH)},
        extraction_options=extraction_options)


In [ ]:
# stereo rig from the factory calibration; opencv convention x_right = R @ x_left + t
# with R = rodrigues([RX, CV, RZ]) and t = [-Baseline, TY, TZ] mm
def zed_rig_config(run, conf):
    cur, st = None, {}
    for line in conf.splitlines():
        line = line.strip()
        if line.startswith('['):
            cur = line.strip('[]')
        elif '=' in line and cur == 'STEREO':
            k, v = line.split('=')
            st[k] = float(v)
    rot = pycolmap.Rotation3d(np.array([st['RX_HD'], st['CV_HD'], st['RZ_HD']]))
    t = np.array([-st['Baseline'], st.get('TY', 0.0), st.get('TZ', 0.0)]) / 1000.0
    return pycolmap.RigConfig(cameras=[
        pycolmap.RigConfigCamera(image_prefix=f'{run}/left/', ref_sensor=True),
        pycolmap.RigConfigCamera(image_prefix=f'{run}/right/', cam_from_rig=pycolmap.Rigid3d(rot, t)),
    ])

if robo:
    with pycolmap.Database.open(str(database)) as db:
        pycolmap.apply_rig_config(
            [zed_rig_config(run, robo[run]['camera']['factory_calibration_conf']) for run in robo], db)


## Matching

In [ ]:
from itertools import combinations
from scipy.spatial import cKDTree

pairs = set()
for run in runs:
    names = sel[run]['names']
    for i in range(len(names)):
        for j in range(i + 1, min(i + 1 + SEQ_K, len(names))):
            pairs.add((names[i], names[j]))
    if run in robo:  # same-frame stereo + right-eye sequential (cross-run stays left-only)
        right = [n.replace('/left/', '/right/') for n in names]
        for i in range(len(names)):
            pairs.add((names[i], right[i]))
            for j in range(i + 1, min(i + 1 + SEQ_K, len(right))):
                pairs.add((right[i], right[j]))
n_seq = len(pairs)

for a, b in combinations(runs, 2):
    tree = cKDTree(sel[b]['enu'][:, :2])
    dists, nbrs = tree.query(sel[a]['enu'][:, :2], k=CROSS_K, distance_upper_bound=CROSS_R)
    for i, (ds, js) in enumerate(zip(dists, nbrs)):
        for d, j in zip(ds, js):
            if not np.isfinite(d):
                continue
            dh = abs(sel[a]['heading'][i] - sel[b]['heading'][j])
            if min(dh, 2 * np.pi - dh) <= np.radians(HEADING_MAX_DEG):
                pairs.add(tuple(sorted((sel[a]['names'][i], sel[b]['names'][j]))))

sfm_pairs.write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(f'{n_seq} sequential + {len(pairs) - n_seq} cross-run pairs')

In [ ]:
matching_options = pycolmap.FeatureMatchingOptions()
matching_options.type = pycolmap.FeatureMatcherType.ALIKED_LIGHTGLUE
matching_options.aliked.lightglue.model_path = str(MODELS_PATH / 'aliked-lightglue.onnx')
matching_options.use_gpu = True
matching_options.gpu_index = GPU_INDEX

pairing_options = pycolmap.ImportedPairingOptions()
pairing_options.match_list_path = str(sfm_pairs)

pycolmap.match_image_pairs(database, matching_options=matching_options,
                           pairing_options=pairing_options)

## Reconstruct

### Database

In [ ]:
PRIOR_STD_XY = 0.5
PRIOR_STD_Z = 1.0
cov = np.diag([PRIOR_STD_XY**2, PRIOR_STD_XY**2, PRIOR_STD_Z**2])

# inert unless use_prior_position is set, so both reconstructions can share the db
with pycolmap.Database.open(str(database)) as db:
    assert db.num_pose_priors() == 0, "priors already written"
    for image in db.read_all_images():
        p = Path(image.name)
        run = p.parts[0]
        if run in robo and p.parts[1] == 'right':
            continue  # frame is constrained via the rig + left-eye prior
        pos_ts, enu = run_enu[run]
        ts = int(p.stem)
        prior = pycolmap.PosePrior()
        prior.corr_data_id = image.data_id
        prior.position = np.array([np.interp(ts, pos_ts, enu[:, i]) for i in range(3)])
        if run in robo:
            # per-frame EKF variance (diagonal approx), floored
            pv = np.array(robo[run]['ekf']['pos_var'])
            var = np.array([np.interp(ts, pos_ts, pv[:, i]) for i in range(3)])
            prior.position_covariance = np.diag(np.maximum(var, ROBO_PRIOR_VAR_FLOOR))
        else:
            prior.position_covariance = cov
        prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
        db.write_pose_prior(prior)
    print('pose priors:', db.num_pose_priors())


### No Prior

In [ ]:
model = reconstruction.run_reconstruction(
    sfm_dir, database, IMAGES_PATH, verbose=True,
    options={
        "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
        "ba_global_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
    })

In [ ]:
p = outputs / 'no_prior'
p.mkdir(parents=True, exist_ok=True)
model.write(p)

### Prior

In [ ]:
# priors only constrain global BA, and caspar doesn't support them
model_prior = reconstruction.run_reconstruction(
    sfm_prior_dir, database, IMAGES_PATH, verbose=True,
    options={
        "use_prior_position": True,
        "use_robust_loss_on_prior_position": True,
        # "ba_use_gpu": True,
        # "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
        # "ba_global_backend": pycolmap.BundleAdjustmentBackend.CERES,
        # Speed options
        # "ba_global_frames_ratio": 1.3,
        # "ba_global_points_ratio": 1.3,
        # "ba_local_max_num_iterations": 12,
        # "ba_local_max_refinements": 2,
        # "ba_global_max_num_iterations": 30,
        # "ba_global_max_refinements": 3,
        # "mapper": {"ba_global_ignore_redundant_points3D": True},
    })

In [ ]:
p = outputs / 'prior'
p.mkdir(parents=True, exist_ok=True)
model_prior.write(p)

# Visualize

### No Prior

In [13]:
model = pycolmap.Reconstruction(str(DATA_PATH / 'outputs' / 'no_prior'))

In [15]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, model, points_rgb=True)
fig.show()

### Prior

In [ ]:
model_prior = pycolmap.Reconstruction(str(DATA_PATH / 'outputs' / 'prior'))

In [ ]:
with pycolmap.Database.open(str(database)) as db:
    priors = db.read_all_pose_priors()
enu = {p.corr_data_id.id: p.position for p in priors}  # already ENU
errs = np.array([model_prior.images[i].projection_center() - enu[i]
                 for i in model_prior.reg_image_ids()])
print(f"rmse vs gps: {np.sqrt((errs[:, :2] ** 2).sum(1).mean()):.2f} m horizontal, "
      f"{np.sqrt((errs[:, 2] ** 2).mean()):.2f} m vertical")

In [ ]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, model_prior, points_rgb=True)
fig.show()